# Amparo -- M2: Evaluacion del modelo fine-tuneado

Este notebook corre el harness de evaluacion de M2 sobre el adaptador LoRA que
entreno M1 (`m1_finetune.ipynb`). Vuelve a generar las respuestas baseline y
fine-tuned sobre el **mismo split de validacion** de M1 (213 ejemplos, seed 42) y
sobre el **eval set propio** (`data/eval_set.json`: 50 gold + 6 adversariales, cada
uno con su criterio).

Fases:
1. Generar respuestas del modelo base (baseline) y del fine-tuned.
2. Quitar el adaptador y usar el modelo base como juez local (1-5 por criterio) +
   comparacion cara a cara.
3. Metricas clasicas, metrica de citas y sesgos.
4. Guardar todo lo que uso la GPU en Drive (si Groq falla despues, no se pierde).
5. Juez externo (Groq, otra familia): comparacion cara a cara y **eval set contra
   su criterio**.
6. Scorecard final y archivos para comitear en `results/`.

**Como leer el resultado** (detalle en la seccion "Que mide cada metrica" del
scorecard): el juez local es el mismo modelo que escribio el baseline; el
compuesto 1-5 incluye concision, que premia la respuesta corta; las metricas
lexicas miden parecido de estilo con la referencia; y el 100 % "sin citas" del
fine-tuned es esperable porque el dataset no cita ninguna norma. Por eso el
scorecard reporta cada criterio con su intervalo de confianza, el conteo de la
comparacion cara a cara con los dos jueces y el eval set contra su criterio.

Toda la logica vive en `tools/evaluation/` (con tests); este notebook solo
orquesta. Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de
ejecucion > GPU`.

In [ ]:
# Rama del repo de la que se toma el codigo y el dataset. M2 evalua el modelo de
# M1, asi que tiene que correr con EL MISMO dataset con el que se entreno: si se
# clona otra rama, la evaluacion mide contra datos distintos y nada falla a la
# vista.
RAMA = "m3.5"

import os

if not os.path.isdir("/content/Amparo"):
    !git clone -q -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd /content/Amparo && git fetch -q origin && git checkout -q {RAMA} && git pull -q
%cd /content/Amparo

!git log -1 --format="Codigo del commit %h (%ad)" --date=short


In [ ]:
# Dependencias puras de tools/evaluation (metricas clasicas, tests)
!pip install -q -r requirements.txt


In [ ]:
# Stack de ML pesado -- igual que en M1 (m1_finetune.ipynb, celda 1):
# se instala aqui y NO en requirements.txt del repo, para no arriesgar
# reemplazar el build de PyTorch con CUDA que Colab ya trae preinstalado.
# bert-score va aqui tambien (no en requirements.txt) porque depende de
# torch de forma transitiva.
!pip install -q -U transformers peft bitsandbytes accelerate bert-score
!pip install -U "bitsandbytes>=0.46.1"


## Configuracion

Las constantes (seed, val_fraction, modelo base, rutas de Drive) viven en
`tools/evaluation/config.py` -- deben coincidir con `RANDOM_SEED=42` y
`VAL_FRACTION=0.15` de M1 para evaluar sobre el mismo split.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

from tools.evaluation import config, dataset

records = dataset.load_records()
train_records, val_records = dataset.stratified_split(records)
system_prompt = dataset.system_prompt(records)

print(f"Total: {len(records)} | train: {len(train_records)} | val: {len(val_records)}")

# El split de M2 tiene que ser el MISMO que uso M1, o la comparacion baseline vs
# afinado deja de ser valida. Se verifica contra el tamano esperado del dataset
# actual (1410 ejemplos -> 1197/213 con RANDOM_SEED=42 y VAL_FRACTION=0.15).
assert len(records) == 1410, (
    f"El dataset tiene {len(records)} ejemplos y se esperaban 1410: "
    "probablemente se clono una rama con el dataset anterior."
)
assert len(val_records) == 213, "El split no coincide con el de M1 -- revisar RANDOM_SEED/VAL_FRACTION"
print("Split verificado: coincide con el usado en M1.")


## Eval set propio (gold + adversariales)

`data/eval_set.json`: 50 gold y 6 adversariales, cada uno con un `criterio` que dice
que debe y que no debe hacer la respuesta. Se califica contra ese criterio en la
Fase 5 (juez Groq). La celda verifica que ninguna pregunta este en train (seria
fuga) e informa cuantas repiten preguntas de validacion: hoy 20 de las 50 gold
(9011-9030), que por eso no cuentan como casos propios.

In [ ]:
from tools.evaluation import eval_set as eval_set_module

eval_records = eval_set_module.load_eval_set()
n_gold = len(eval_set_module.gold_examples(eval_records))
n_adv = len(eval_set_module.adversarial_examples(eval_records))
print(f"Eval set: {len(eval_records)} ejemplos ({n_gold} gold, {n_adv} adversariales)")

solape = eval_set_module.solapamiento(eval_records, train_records, val_records)
assert not solape["en_train"], f"Fuga: preguntas del eval set en train: {solape['en_train']}"
print(f"Sin fuga a train. Repiten preguntas de validacion: {len(solape['en_val'])} -> {solape['en_val']}")

## Fase 1 -- Generacion: baseline (modelo sin fine-tuning)

Con `MAX_NEW_TOKENS_GENERATION = 900` (antes 300, que cortaba el 75 % de las
respuestas del baseline). Se imprime cuantas quedaron cortadas.

In [ ]:
from tools.evaluation import generation

model, tokenizer = generation.load_base_model()

baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="baseline"
)
print(f"Generadas {len(baseline_results)} respuestas baseline; cortadas por max_new_tokens: {sum(r.cortada for r in baseline_results)}.")


In [ ]:
eval_set_baseline_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="baseline"
)
print(f"Eval set (baseline): {len(eval_set_baseline_results)} respuestas; cortadas: {sum(r.cortada for r in eval_set_baseline_results)}.")


## Fase 1b -- Generacion: fine-tuned (adaptador LoRA desde Drive)


In [ ]:
model = generation.attach_adapter(model, config.DRIVE_ADAPTER_DIR)
model.eval()

finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, val_records, label="fine_tuned"
)
print(f"Generadas {len(finetuned_results)} respuestas fine-tuned; cortadas: {sum(r.cortada for r in finetuned_results)}.")


In [ ]:
eval_set_finetuned_results = generation.generate_batch(
    model, tokenizer, system_prompt, eval_records, label="fine_tuned"
)
print(f"Eval set (fine-tuned): {len(eval_set_finetuned_results)} respuestas; cortadas: {sum(r.cortada for r in eval_set_finetuned_results)}.")


## Fase 2 -- Juez local (LLM-as-a-Judge)

### Liberar el adaptador y reutilizar el modelo base como juez

`detach_adapter` usa `model.unload()` (quita las capas LoRA sin fusionarlas)
-- así el mismo modelo cargado en memoria sirve como juez independiente del
adaptador, sin una segunda carga completa de modelo.


In [ ]:
import torch

model = generation.detach_adapter(model)
torch.cuda.empty_cache()
print("Adaptador liberado. El modelo en memoria es ahora el juez (base, sin fine-tuning).")


### Puntuacion 1-5 y comparacion cara a cara

Puntuacion 1-5 por criterio (correccion juridica, prudencia, claridad, concision)
de cada respuesta contra la referencia del dataset, y comparacion cara a cara en
una muestra de 30 pares, cada par en los dos ordenes.

**Ojo:** este juez es el mismo modelo base que escribio las respuestas del
baseline, asi que puede preferirlas (auto-preferencia). En la corrida del
2026-10-02 eligio al baseline en 50 de 60 veredictos, mientras el compuesto 1-5
casi no distinguia a los dos modelos. Por eso la Fase 5 repite la comparacion con
un juez de otra familia. Leer siempre el conteo de ganadores, no solo el flip
rate: si un modelo gana en los dos ordenes, el flip rate es 0 % y eso no es
neutralidad.

In [ ]:
from tools.evaluation import judge

judge_baseline = judge.score_batch(model, tokenizer, baseline_results)
judge_finetuned = judge.score_batch(model, tokenizer, finetuned_results)

n_fail_base = sum(1 for s in judge_baseline if not s.parse_ok)
n_fail_ft = sum(1 for s in judge_finetuned if not s.parse_ok)
print(f"Juez baseline: {len(judge_baseline)} filas, {n_fail_base} fallos de parseo.")
print(f"Juez fine-tuned: {len(judge_finetuned)} filas, {n_fail_ft} fallos de parseo.")


In [ ]:
from tools.evaluation import bias

pairs = [
    (b.id, b.query, b.generated, f.generated)
    for b, f in zip(baseline_results, finetuned_results)
]
position_bias_report = bias.run_position_bias_probe(model, tokenizer, pairs)
print(f"Flip rate: {position_bias_report.flip_rate_pct}% ({position_bias_report.n_pairs} pares)")
print("Conteo de veredictos (juez local):", position_bias_report.winner_counts())

## Fase 3 -- Metricas clasicas, metrica de citas y sesgos

No requieren GPU (BERTScore descarga un modelo aparte, chico).

In [ ]:
from tools.evaluation import pipeline

eval_rows_baseline = pipeline.build_eval_rows(baseline_results, judge_baseline)
eval_rows_finetuned = pipeline.build_eval_rows(finetuned_results, judge_finetuned)

pipeline.fill_bertscore(eval_rows_baseline)
pipeline.fill_bertscore(eval_rows_finetuned)

all_rows = eval_rows_baseline + eval_rows_finetuned
print(f"Total de filas evaluadas: {len(all_rows)}")


In [ ]:
length_bias_baseline = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_baseline if r.judge_composite is not None],
)
length_bias_finetuned = bias.length_bias_correlation(
    [r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    [len(r.generated) for r in eval_rows_finetuned if r.judge_composite is not None],
)

self_pref_report = bias.self_preference_gap(
    judge_baseline=[r.judge_composite for r in eval_rows_baseline if r.judge_composite is not None],
    judge_finetuned=[r.judge_composite for r in eval_rows_finetuned if r.judge_composite is not None],
    sim_baseline=[r.similarity_pct for r in eval_rows_baseline if r.similarity_pct is not None],
    sim_finetuned=[r.similarity_pct for r in eval_rows_finetuned if r.similarity_pct is not None],
)

bias_summary = {
    "length_bias_pearson_r_baseline": length_bias_baseline["pearson_r"],
    "length_bias_pearson_r_fine_tuned": length_bias_finetuned["pearson_r"],
    "self_preference_judge_gap": self_pref_report.judge_gap,
    "self_preference_similarity_gap": self_pref_report.similarity_gap,
    "self_preference_divergence": self_pref_report.divergence,
    "self_preference_flagged": self_pref_report.flagged,
    "position_bias_flip_rate_pct": position_bias_report.flip_rate_pct,
    "position_bias_n_pairs": position_bias_report.n_pairs,
}
bias_summary


## Fase 4 -- Guardar lo que uso la GPU (Drive)

Cada corrida va a su propia carpeta con timestamp. Se guardan las respuestas
(con el flag de cortada), las metricas por registro, el sondeo del juez local y
un scorecard preliminar. Si la Fase 5 (Groq) falla o se acaba el cupo, nada de
esto se pierde y la Fase 5 se puede retomar despues (los checkpoints solo reusan
resultados del mismo texto).

In [ ]:
import json as _json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from tools.evaluation import scorecard

manifest = pipeline.build_manifest(n_val=len(val_records), hardware=subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip())

run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
drive_run_dir = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/{run_id}")
drive_run_dir.mkdir(parents=True, exist_ok=True)


def _guardar_jsonl(nombre, filas):
    (drive_run_dir / nombre).write_text(
        "\n".join(_json.dumps(r.__dict__, ensure_ascii=False) for r in filas) + "\n", encoding="utf-8")


(drive_run_dir / "run_manifest.json").write_text(
    _json.dumps(manifest.__dict__, indent=2, ensure_ascii=False), encoding="utf-8")
_guardar_jsonl("resultados_baseline.jsonl", baseline_results)
_guardar_jsonl("resultados_finetuned.jsonl", finetuned_results)
_guardar_jsonl("eval_set_baseline_results.jsonl", eval_set_baseline_results)
_guardar_jsonl("eval_set_finetuned_results.jsonl", eval_set_finetuned_results)
(drive_run_dir / "position_bias_probe.json").write_text(
    _json.dumps(position_bias_report.__dict__, indent=2, ensure_ascii=False), encoding="utf-8")
scorecard.export_csv(drive_run_dir / "metricas_por_registro.csv", all_rows)

summaries = scorecard.summarize_by_label(all_rows)
comparacion = scorecard.comparar(all_rows)
category_summaries = {
    "baseline": scorecard.summarize_by_category(all_rows, "baseline"),
    "fine_tuned": scorecard.summarize_by_category(all_rows, "fine_tuned"),
}
ejemplos = scorecard.elegir_ejemplos(eval_records, eval_set_baseline_results, eval_set_finetuned_results)
ganadores = {"Qwen2.5-7B (juez local, mismo modelo que el baseline)": position_bias_report.winner_counts()}


def escribir_scorecard(eval_set_resumen=None, juez_externo=None):
    narrativa = scorecard.build_narrative(summaries, bias_summary, n_val=len(val_records),
                                          comparacion=comparacion, ganadores=ganadores,
                                          eval_set=eval_set_resumen)
    scorecard.export_markdown(
        drive_run_dir / "scorecard.md", summaries, category_summaries, narrativa, bias_summary,
        manifest.__dict__, comparacion=comparacion, ganadores=ganadores, eval_set=eval_set_resumen,
        ejemplos=ejemplos, juez_externo=juez_externo,
    )
    return (drive_run_dir / "scorecard.md").read_text(encoding="utf-8")


escribir_scorecard()
print(f"Guardado en: {drive_run_dir} (scorecard preliminar, sin Groq)")

## (Opcional) Explorar resultados en pandas


In [ ]:
import pandas as pd

df = pd.DataFrame([{**r.__dict__, "judge_sin_concision": r.judge_sin_concision, "n_palabras": r.n_palabras}
                   for r in all_rows])
df.groupby("label")[["judge_correccion", "judge_prudencia", "judge_claridad", "judge_concision",
                     "judge_sin_concision", "n_palabras", "cortada", "citation_count"]].mean()

## Fase 5 -- Juez externo (Groq): comparacion cara a cara y eval set contra su criterio

Groq (`openai/gpt-oss-120b`) es de otra familia que Qwen: no comparte el sesgo de
auto-preferencia del juez local. Aqui:

1. Repite la comparacion cara a cara sobre los MISMOS 30 pares.
2. Califica el eval set **contra su criterio**: cumple / parcial / no cumple, y la
   lista de errores juridicos concretos (entidades equivocadas o inexistentes,
   afirmaciones falsas), que ninguna metrica automatica detecta. La guardia de
   citas numeradas tambien se aplica al eval set.
3. (Opcional) Califica la validacion completa 1-5 con Groq, para tener el juez
   principal sin auto-preferencia. Son ~430 llamadas: no caben en el cupo gratuito
   de un dia (~200.000 tokens), asi que se retoma al dia siguiente o con la key de
   otra cuenta.

Requiere `GROQ_API_KEY` en Colab Secrets. Los checkpoints van en
`evaluacion/checkpoints/` y solo se reusan si el texto calificado es el mismo: si
la corrida se corta, volver a correr la celda sigue donde quedo. Si se cambia la
key, volver a correr la celda siguiente (reinicia el cliente).

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from tools.evaluation import criterio, external_judge

external_judge._client = None   # si se cambio la key, el cliente se recrea con la nueva
CHECKPOINT_DIR = Path(f"{config.DRIVE_EVAL_OUTPUT_ROOT}/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
groq_position_bias_report = external_judge.run_position_bias_probe(
    pairs, checkpoint_path=CHECKPOINT_DIR / "groq_position_bias.jsonl"
)
print(f"Flip rate (Groq): {groq_position_bias_report.flip_rate_pct}%")
print("Conteo de veredictos (Groq):", groq_position_bias_report.winner_counts())
print("Conteo de veredictos (Qwen):", position_bias_report.winner_counts())
ganadores["Groq gpt-oss-120b (otra familia)"] = groq_position_bias_report.winner_counts()

In [ ]:
juez_groq = criterio.generador_groq()
veredictos_base = criterio.evaluar_contra_criterio(
    eval_set_baseline_results, eval_records, juez_groq,
    checkpoint_path=CHECKPOINT_DIR / "groq_criterio_eval_set_baseline.jsonl")
veredictos_ft = criterio.evaluar_contra_criterio(
    eval_set_finetuned_results, eval_records, juez_groq,
    checkpoint_path=CHECKPOINT_DIR / "groq_criterio_eval_set_finetuned.jsonl")
eval_set_resumen = criterio.resumen(veredictos_base + veredictos_ft)

print("=== Eval set contra su criterio (juez Groq) ===")
for rec, gb, gf, vb, vf in zip(eval_records, eval_set_baseline_results, eval_set_finetuned_results,
                                veredictos_base, veredictos_ft):
    print(f"\n--- id {rec['id']} ({rec['tipo']}) {rec['category']} ---")
    print(f"Pregunta: {rec['messages'][1]['content']}")
    print(f"Criterio: {rec['criterio']}")
    print(f"[baseline]   {gb.generated}")
    print(f"  -> {vb.veredicto} | errores: {vb.errores_juridicos} | citas numeradas: {vb.n_citas}")
    print(f"[fine-tuned] {gf.generated}")
    print(f"  -> {vf.veredicto} | errores: {vf.errores_juridicos} | citas numeradas: {vf.n_citas}")

print("\n", _json.dumps(eval_set_resumen, indent=2, ensure_ascii=False))

In [ ]:
# (Opcional) Validacion completa 1-5 con Groq. ~430 llamadas: se retoma por dias o
# con otra cuenta (checkpoint con huella). Dejar en False si no hay cupo.
CALIFICAR_VALIDACION_CON_GROQ = False

juez_externo = None
if CALIFICAR_VALIDACION_CON_GROQ:
    groq_base = external_judge.score_batch(
        baseline_results, checkpoint_path=CHECKPOINT_DIR / "groq_validacion_baseline.jsonl")
    groq_ft = external_judge.score_batch(
        finetuned_results, checkpoint_path=CHECKPOINT_DIR / "groq_validacion_finetuned.jsonl")
    filas_groq = (pipeline.build_eval_rows(baseline_results, groq_base)
                  + pipeline.build_eval_rows(finetuned_results, groq_ft))
    sin_parseo = sum(1 for r in filas_groq if not r.judge_parse_ok)
    print(f"Groq: {len(filas_groq)} filas, {sin_parseo} sin calificar (cupo o parseo).")
    if sin_parseo == 0:
        juez_externo = ("Groq gpt-oss-120b (validacion completa)",
                        scorecard.summarize_by_label(filas_groq), scorecard.comparar(filas_groq))
    else:
        print("Incompleto: volver a correr esta celda cuando haya cupo (sigue donde quedo).")

## Fase 6 -- Scorecard final y archivos para el repo

Regenera el scorecard con todo (juez local, Groq, eval set contra criterio) y
guarda en la carpeta de la corrida los archivos que van a `results/`:

- `scorecard.md`, `metricas_por_registro.csv`, `run_manifest.json`
- `eval_set_resultados.jsonl` (respuestas + veredicto contra el criterio + errores)
- `position_bias_probe.json` y `groq_position_bias_probe.json`

**Para comitear:** descargar la carpeta de la corrida de Drive y copiar esos
archivos a `results/m2_<fecha>/` en el repo. Los resultados no pueden vivir solo
en Drive (revision docente de M2).

In [ ]:
(drive_run_dir / "groq_position_bias_probe.json").write_text(
    _json.dumps(groq_position_bias_report.__dict__, indent=2, ensure_ascii=False), encoding="utf-8")
(drive_run_dir / "eval_set_resultados.jsonl").write_text("\n".join(
    _json.dumps({
        "id": rec["id"], "tipo": rec["tipo"], "category": rec["category"], "criterio": rec["criterio"],
        "query": rec["messages"][1]["content"],
        "baseline": gb.generated, "baseline_cortada": gb.cortada,
        "baseline_veredicto": vb.veredicto, "baseline_errores": vb.errores_juridicos,
        "baseline_citas": vb.n_citas, "baseline_justificacion": vb.justificacion,
        "fine_tuned": gf.generated, "fine_tuned_cortada": gf.cortada,
        "fine_tuned_veredicto": vf.veredicto, "fine_tuned_errores": vf.errores_juridicos,
        "fine_tuned_citas": vf.n_citas, "fine_tuned_justificacion": vf.justificacion,
    }, ensure_ascii=False)
    for rec, gb, gf, vb, vf in zip(eval_records, eval_set_baseline_results, eval_set_finetuned_results,
                                    veredictos_base, veredictos_ft)) + "\n", encoding="utf-8")

print(escribir_scorecard(eval_set_resumen=eval_set_resumen, juez_externo=juez_externo))
print(f"\nArchivos para results/m2_<fecha>/ en: {drive_run_dir}")